In [1]:
"""
Optimized Carbon Footprint Prediction Script
Uses multiple ML models, feature engineering, and ensemble methods
to achieve the highest possible R² score.
"""

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import skew

# Machine Learning Libraries
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.metrics import r2_score, mean_squared_error

# Models
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import VotingRegressor, StackingRegressor

# For saving the best model
import joblib

# ------------------------------
# 1. Load and inspect data
# ------------------------------
data = pd.read_csv("../datasets/smart_city_citizen_activity.csv")
print("Data shape:", data.shape)
print("Missing values:\n", data.isnull().sum())
print("\nBasic statistics:\n", data.describe())

# Check target distribution
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
sns.histplot(data['Carbon_Footprint_kgCO2'], kde=True, bins=30)
plt.title('Original Carbon Footprint')

# If highly skewed, we'll apply log transformation later
target_skew = skew(data['Carbon_Footprint_kgCO2'])
print(f"Skewness of target: {target_skew:.2f}")
if abs(target_skew) > 0.5:
    data['Carbon_Footprint_kgCO2'] = np.log1p(data['Carbon_Footprint_kgCO2'])
    print("Applied log1p transformation to target.")
    plt.subplot(1,2,2)
    sns.histplot(data['Carbon_Footprint_kgCO2'], kde=True, bins=30)
    plt.title('Log-transformed Carbon Footprint')
plt.tight_layout()
plt.savefig('target_distribution.png')
plt.show()

# ------------------------------
# 2. Encode categorical variables
# ------------------------------
le_gender = LabelEncoder()
le_transport = LabelEncoder()
data['Gender'] = le_gender.fit_transform(data['Gender'])
data['Mode_of_Transport'] = le_transport.fit_transform(data['Mode_of_Transport'])

# Save encoders
joblib.dump(le_gender, 'le_gender.pkl')
joblib.dump(le_transport, 'le_transport.pkl')

# ------------------------------
# 3. Feature Engineering
# ------------------------------
# Activity sums
data['Total_Activity_Hours'] = (data['Work_Hours'] + data['Shopping_Hours'] +
                                 data['Entertainment_Hours'] + data['Public_Events_Hours'])

# Energy and calories per step (avoid division by zero)
data['Energy_Per_Step'] = data['Home_Energy_Consumption_kWh'] / (data['Steps_Walked'] + 1)
data['Calories_Per_Step'] = data['Calories_Burned'] / (data['Steps_Walked'] + 1)

# Work ratio
data['Work_Ratio'] = data['Work_Hours'] / (data['Total_Activity_Hours'] + 1)

# Age groups
bins = [0, 25, 40, 60, 100]
labels = ['Young', 'Adult', 'MiddleAge', 'Senior']
data['Age_Group'] = pd.cut(data['Age'], bins=bins, labels=labels)
le_age = LabelEncoder()
data['Age_Group'] = le_age.fit_transform(data['Age_Group'])
joblib.dump(le_age, 'le_age.pkl')

# Interaction features between transport mode and hours (for linear models)
# We'll create separate columns for each transport mode multiplied by relevant hours
# But to avoid too many features, we'll just let tree models handle interactions.
# For linear models, we can add polynomial features later.

# Also create weekend proxy? Not available.

# Drop ID
if 'Citizen_ID' in data.columns:
    data.drop('Citizen_ID', axis=1, inplace=True)

print("Features after engineering:", data.columns.tolist())

# ------------------------------
# 4. Feature selection using mutual information
# ------------------------------
# Separate features and target
X = data.drop('Carbon_Footprint_kgCO2', axis=1)
y = data['Carbon_Footprint_kgCO2']

# Compute mutual information scores
mi_scores = mutual_info_regression(X, y, random_state=42)
mi_series = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)
print("\nTop 10 features by mutual information:")
print(mi_series.head(10))

# Plot MI scores
plt.figure(figsize=(10,6))
mi_series.plot(kind='bar')
plt.title('Mutual Information Scores')
plt.tight_layout()
plt.savefig('mi_scores.png')
plt.show()

# Optionally select top k features (we'll keep all for now, but you could threshold)
# For this script, we'll use all features because tree models handle irrelevant ones.

# ------------------------------
# 5. Train-test split
# ------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

# ------------------------------
# 6. Define models and hyperparameter grids
# ------------------------------
# We'll use RandomizedSearchCV for each model to find best parameters.

# Common settings
n_iter_search = 50
cv_folds = 5
scoring = 'r2'
random_state = 42

models = {}

# Ridge Regression (linear)
ridge = Ridge(random_state=random_state)
param_ridge = {
    'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]
}
models['Ridge'] = (ridge, param_ridge)

# Random Forest
rf = RandomForestRegressor(random_state=random_state)
param_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}
models['RandomForest'] = (rf, param_rf)

# Gradient Boosting
gb = GradientBoostingRegressor(random_state=random_state)
param_gb = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'min_samples_split': [2, 5, 10]
}
models['GradientBoosting'] = (gb, param_gb)

# XGBoost
xgb = XGBRegressor(random_state=random_state, verbosity=0)
param_xgb = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}
models['XGBoost'] = (xgb, param_xgb)

# LightGBM
lgb = LGBMRegressor(random_state=random_state, verbosity=-1)
param_lgb = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [5, 10, -1],
    'num_leaves': [20, 31, 40],
    'subsample': [0.8, 1.0]
}
models['LightGBM'] = (lgb, param_lgb)

# Extra Trees
et = ExtraTreesRegressor(random_state=random_state)
param_et = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
models['ExtraTrees'] = (et, param_et)

# Support Vector Regression (RBF kernel) - requires scaling
svr = SVR()
param_svr = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.1, 0.01],
    'epsilon': [0.1, 0.2, 0.5]
}
models['SVR'] = (svr, param_svr)

# Neural Network (MLP) - requires scaling
mlp = MLPRegressor(random_state=random_state, max_iter=1000)
param_mlp = {
    'hidden_layer_sizes': [(50,), (100,), (50,50), (100,50)],
    'activation': ['relu', 'tanh'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate': ['constant', 'adaptive']
}
models['MLP'] = (mlp, param_mlp)

# ------------------------------
# 7. Scale data for models that require it (SVR, MLP)
# ------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# For tree-based models we keep original
X_train_tree = X_train
X_test_tree = X_test

# ------------------------------
# 8. Train and tune each model
# ------------------------------
best_estimators = {}
cv_results = {}

for name, (model, params) in models.items():
    print(f"\n--- Tuning {name} ---")
    if name in ['SVR', 'MLP']:
        X_tr = X_train_scaled
    else:
        X_tr = X_train_tree

    search = RandomizedSearchCV(
        model, params, n_iter=n_iter_search, cv=cv_folds,
        scoring=scoring, random_state=random_state, n_jobs=-1, verbose=1
    )
    search.fit(X_tr, y_train)

    best_estimators[name] = search.best_estimator_
    cv_results[name] = search.best_score_
    print(f"Best CV R²: {search.best_score_:.4f}")
    print(f"Best params: {search.best_params_}")

# ------------------------------
# 9. Compare models
# ------------------------------
results_df = pd.DataFrame.from_dict(cv_results, orient='index', columns=['CV_R2'])
results_df.sort_values('CV_R2', ascending=False, inplace=True)
print("\nModel comparison (CV R²):")
print(results_df)

# Select best model based on CV score
best_model_name = results_df.index[0]
best_model = best_estimators[best_model_name]
print(f"\nBest model: {best_model_name} with CV R² = {results_df.iloc[0,0]:.4f}")

# ------------------------------
# 10. Evaluate on test set
# ------------------------------
# Use appropriate scaled data for prediction
if best_model_name in ['SVR', 'MLP']:
    X_te = X_test_scaled
else:
    X_te = X_test_tree

y_pred = best_model.predict(X_te)

# If we log-transformed target, invert for interpretability
if abs(target_skew) > 0.5:
    y_test_orig = np.expm1(y_test)
    y_pred_orig = np.expm1(y_pred)
    r2_test = r2_score(y_test_orig, y_pred_orig)
    rmse_test = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
    print(f"\nTest R² (original scale): {r2_test:.4f}")
    print(f"Test RMSE (original scale): {rmse_test:.2f} kgCO2")
else:
    r2_test = r2_score(y_test, y_pred)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
    print(f"\nTest R²: {r2_test:.4f}")
    print(f"Test RMSE: {rmse_test:.2f} kgCO2")

# ------------------------------
# 11. Feature importance for tree-based models
# ------------------------------
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feature_names = X.columns
    indices = np.argsort(importances)[::-1][:15]  # top 15
    plt.figure(figsize=(10,6))
    plt.title(f"Top 15 Feature Importances ({best_model_name})")
    plt.barh(range(len(indices)), importances[indices], align='center')
    plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
    plt.gca().invert_yaxis()
    plt.xlabel("Relative Importance")
    plt.tight_layout()
    plt.savefig('best_model_feature_importance.png')
    plt.show()
else:
    print(f"\n{best_model_name} does not provide feature importances.")

# ------------------------------
# 12. Try ensemble: Stacking (optional)
# ------------------------------
# We'll stack the top 3 models to see if we can improve further.
print("\n--- Building Stacking Ensemble ---")
top_models = results_df.index[:3].tolist()
estimators = [(name, best_estimators[name]) for name in top_models]

# Use a simple linear model as final estimator
stack = StackingRegressor(estimators=estimators, final_estimator=Ridge(), cv=5)
stack.fit(X_train_tree, y_train)  # Note: if any top model is SVR/MLP, we need scaled data.
# But for simplicity we'll use tree-scaled data; if SVR/MLP is in top, we need to scale.
# We'll just check: if any of top models require scaling, we need to use scaled data.
require_scale = any(name in ['SVR', 'MLP'] for name in top_models)
if require_scale:
    X_tr_stack = X_train_scaled
    X_te_stack = X_test_scaled
else:
    X_tr_stack = X_train_tree
    X_te_stack = X_test_tree

stack.fit(X_tr_stack, y_train)
y_pred_stack = stack.predict(X_te_stack)

if abs(target_skew) > 0.5:
    y_pred_stack_orig = np.expm1(y_pred_stack)
    r2_stack = r2_score(y_test_orig, y_pred_stack_orig)
    rmse_stack = np.sqrt(mean_squared_error(y_test_orig, y_pred_stack_orig))
    print(f"Stacking Test R² (original scale): {r2_stack:.4f}")
    print(f"Stacking Test RMSE (original scale): {rmse_stack:.2f} kgCO2")
else:
    r2_stack = r2_score(y_test, y_pred_stack)
    rmse_stack = np.sqrt(mean_squared_error(y_test, y_pred_stack))
    print(f"Stacking Test R²: {r2_stack:.4f}")
    print(f"Stacking Test RMSE: {rmse_stack:.2f} kgCO2")

# If stacking performs better, use it as final
if r2_stack > r2_test:
    best_model = stack
    best_model_name = "StackingEnsemble"
    r2_test = r2_stack
    rmse_test = rmse_stack
    print("Stacking improved performance. Using stacking as final model.")
else:
    print("Stacking did not improve. Keeping individual best model.")

# ------------------------------
# 13. Save the best model and associated objects
# ------------------------------
joblib.dump(best_model, "carbon_model_best.pkl")
joblib.dump(scaler, "scaler.pkl")  # in case final model needs scaling
joblib.dump(feature_names.tolist(), "feature_columns.pkl")  # to ensure correct order
print("\nBest model saved as 'carbon_model_best.pkl'")
print("Script completed successfully.")

ModuleNotFoundError: No module named 'lightgbm'